In [10]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
from pathlib import Path
import sqlite3

In [11]:
# Initialization

load_dotenv(dotenv_path="../.env", override=True)

openai_api_key = os.getenv("OPENAI_API_KEY")

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")

MODEL = "gpt-4.1-mini"
client = OpenAI()

OpenAI API Key exists and begins sk-proj-


In [12]:
DB = "timber.db"

def get_available_timber_types():
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
    cursor.execute("SELECT type, unit_price FROM timber_prices")
    rows = cursor.fetchall()

    if not rows:
        return "No timber types are currently available."

    result = "Available timber types:\n"
    for timber_type, unit_price in rows:
        result += f"- {timber_type}: {unit_price:,.0f} RWF per unit\n"

    return result

available_timber_tool = {
    "name": "get_available_timber_types",
    "description": "Get the list of available timber types and their unit prices in RWF",
    "parameters": {
        "type": "object",
        "properties": {},
        "required": [],
        "additionalProperties": False
    }
}

In [13]:
def get_timber_price(timber_type):
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()

    cursor.execute(
        "SELECT unit_price FROM timber_prices WHERE LOWER(type) = LOWER(?)",
        (timber_type,)
    )
    row = cursor.fetchone()

    if row:
        return f"The unit price of {timber_type} is {row[0]:,.0f} RWF."
    else:
        return f"Sorry, {timber_type} is not available."
    

price_tool = {
    "name": "get_timber_price",
    "description": "Get the unit price of a specific timber type in RWF",
    "parameters": {
        "type": "object",
        "properties": {
            "timber_type": {
                "type": "string",
                "description": "The type of timber, for example Teak, Pine, or Cedar"
            }
        },
        "required": ["timber_type"],
        "additionalProperties": False
    }
}

In [14]:
def calculate_timber_cost(timber_type, quantity):
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()

    cursor.execute(
        "SELECT unit_price FROM timber_prices WHERE LOWER(type) = LOWER(?)",
        (timber_type,)
    )
    row = cursor.fetchone()

    if not row:
        return f"Sorry, {timber_type} is not available."

    unit_price = row[0]
    total_price = unit_price * quantity

    return (
        f"{quantity} units of {timber_type} cost "
        f"{total_price:,.0f} RWF "
        f"at {unit_price:,.0f} RWF per unit."
    )


cost_tool = {
    "name": "calculate_timber_cost",
    "description": "Calculate the total price for a quantity of a specific timber type in RWF",
    "parameters": {
        "type": "object",
        "properties": {
            "timber_type": {
                "type": "string",
                "description": "The type of timber, for example Teak, Pine, or Cedar"
            },
            "quantity": {
                "type": "integer",
                "description": "The number of timber units the customer wants to buy"
            }
        },
        "required": ["timber_type", "quantity"],
        "additionalProperties": False
    }
}



In [15]:
tools = [
    {"type": "function", "function": available_timber_tool},
    {"type": "function", "function": price_tool},
    {"type": "function", "function": cost_tool}
]

In [16]:
system_message = """
You are a helpful assistant for a timber company in Rwanda.
Prices are in RWF per timber unit.
Use tools to answer questions about available timber types, unit prices, and total costs.
If the customer asks for a timber type that is not available, say so.
Give short and clear answers.
"""


def handle_tool_call(message):
    responses = []

    tool_map = {
        "get_available_timber_types": lambda args: get_available_timber_types(),
        "get_timber_price": lambda args: get_timber_price(args.get("timber_type")),
        "calculate_timber_cost": lambda args: calculate_timber_cost(
            args.get("timber_type"),
            args.get("quantity")
        ),
    }

    for tool_call in message.tool_calls:
        function_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)

        if function_name in tool_map:
            result = tool_map[function_name](arguments)
        else:
            result = f"Unknown tool: {function_name}"

        responses.append({
            "role": "tool",
            "content": result,
            "tool_call_id": tool_call.id
        })

    return responses


def chat(message, history):
    history = [{"role": h["role"], "content": h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = client.chat.completions.create(
        model = MODEL, messages=messages, tools=tools
    )

    while response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        responses = handle_tool_call(message)
        messages.append(message)
        messages.extend(responses)
        response = client.chat.completions.create(
            model=MODEL, messages=messages, tools=tools
        )
    
    return response.choices[0].message.content


In [19]:

initial_message = [
    {
        "role": "assistant",
        "content": "Hello! How can I assist you with timber products today?"
    }
]
chatbot = gr.Chatbot(value=initial_message)

gr.ChatInterface(
    fn=chat,
    chatbot=chatbot
).launch()

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.
